In [ ]:
!pip install skimpy nltk spacy wordcloud xgboost lightgbm sentence-transformers torch netron skorch mlflow pyngrok imblearn hmmlearn dowhy lime

In [ ]:
import pandas as pd
import numpy as np
import skimpy as sk
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
import mlflow.sklearn
import mlflow.pytorch
from scipy.special import softmax
from imblearn.over_sampling import SMOTE
from imblearn.over_sampling import RandomOverSampler
from pyngrok import ngrok
from scipy.stats import *
from scipy import stats
from dowhy import CausalModel
from sklearn.inspection import permutation_importance
from lime.lime_tabular import LimeTabularExplainer

from sklearn.model_selection import *
from sklearn.preprocessing import *
from sklearn.impute import *
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import *
from sklearn.utils import resample
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from lightgbm import LGBMClassifier, LGBMRegressor
from sklearn.ensemble import StackingClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.isotonic import IsotonicRegression

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import pipeline
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.utils.data as data
import torch.utils.data.dataset as Dataset

In [ ]:
train = pd.read_csv("/kaggle/input/datasets/dataset69420/trigagiest/triagegeist/train.csv").drop(columns=["patient_id",	"site_id", "triage_nurse_id"], axis=1)
test = pd.read_csv("/kaggle/input/datasets/dataset69420/trigagiest/triagegeist/test.csv").drop(columns=["patient_id",	"site_id", "triage_nurse_id"], axis=1)

In [ ]:
train

In [ ]:
test

In [ ]:
sk.skim(train)

In [ ]:
train.isnull().sum().plot(kind="bar")

In [ ]:
test.isnull().sum().plot(kind="bar")

In [ ]:
train.columns.difference(test.columns)

## EDA

In [ ]:
#Estimating Outliers
numeric_train_df = train.select_dtypes(include=np.number)

plt.figure(figsize=(30, 10))
sns.boxplot(data=numeric_train_df)
plt.title('Outlier Distribution')
plt.savefig("Outlier_Distribution.png")
plt.show()

In [ ]:
#Correlation
corr = numeric_train_df.corr()
plt.figure(figsize=(30,10))
sns.heatmap(corr, annot=True, cmap='coolwarm')
plt.title('Correlation Matrix')
plt.savefig("Correlation_Matrix.png")
plt.show()

The code for The pairplots (Visualizations in the repo):

```python
sns.pairplot(numeric_train_df)
plt.savefig("Pairplots.png")
plt.show()
```

In [ ]:
#Class Counts
sns.countplot(x='triage_acuity', data=numeric_train_df)
plt.title('Class Counts')
plt.savefig("Class_Counts.png")
plt.show()

In [ ]:
#Distribution Plot
sns.distplot(train['triage_acuity'], kde=True)
plt.title('Distribution Plot')
plt.savefig("Distribution_Plot.png")

In [ ]:
#Violin Plot
sns.violinplot(x='triage_acuity', y='triage_acuity', data=numeric_train_df)
plt.title('Violin Plot')
plt.savefig("Violin_Plot.png")
plt.show()

In [ ]:
#Disposition Feature Counts
sns.countplot(x='disposition', data=train)
plt.title('Disposition Feature Counts')
plt.savefig("Disposition_Feature_Counts.png")
plt.show()

In [ ]:
#ed_los_hours Continuous Variable Distribution
sns.distplot(train['ed_los_hours'], kde=True)
plt.title('ed_los_hours Continuous Variable Distribution')
plt.savefig("ed_los_hours_Continuous_Variable_Distribution.png")
plt.show()

In [ ]:
!zip -r all_images.zip /content/*.png

#Feature Engineering

In [ ]:
#Label Encoding Variables
labeler = {}

for col in train.columns:
    if train[col].dtype == 'object':
        le = LabelEncoder()

        # Fit on train
        train[col] = le.fit_transform(train[col].astype(str))

        # Store encoder
        labeler[col] = le

for col in test.columns:
    if test[col].dtype == 'object':
        test[col] = labeler[col].transform(test[col].astype(str))

In [ ]:
#Imputing Values for All Missing Values
imputer  = SimpleImputer(missing_values=np.nan, strategy='most_frequent')

for col in train.columns:
  train[col] = imputer.fit_transform(train[col].values.reshape(-1, 1))

for col in test.columns:
  test[col] = imputer.fit_transform(test[col].values.reshape(-1, 1))

In [ ]:
train

In [ ]:
train_df_disp = train.copy()

X_disp = train.drop(columns=['triage_acuity', 'disposition', 'ed_los_hours'], axis=1)
y_disp = train['disposition']

In [ ]:
test_df_disp = test.copy()

In [ ]:
#Making Models for Feature Engineering and Discovering Hidden Latent Regimes
# --- Disposition ---
X_disp_train, X_disp_test, y_disp_train, y_disp_test = train_test_split(X_disp, y_disp, stratify=y_disp, test_size=0.2, random_state=42)

X_disp_train, y_disp_train = RandomOverSampler(random_state=42).fit_resample(X_disp_train, y_disp_train)

class_weight = {
    0: 1.0,
    1: 15.0,   # very rare
    2: 1.0,
    3: 5.0,
    4: 5.0,
    5: 4.0,
    6: 3.0
}

disp_xgb = XGBClassifier(
    objective='multi:softprob',
    class_weight=class_weight,
    num_class=7,
    learning_rate=0.03,
    n_estimators=2000,
    subsample=0.8,
    colsample_bytree=0.8,
    max_depth=6,
    min_child_weight=3,
    reg_alpha=0.6,
    reg_lambda=1.0,
    random_state=42,
    tree_method='hist',
    n_jobs=-1
)

disp_xgb.fit(
    X_disp_train, y_disp_train
)

preds = disp_xgb.predict(X_disp_test)
print(classification_report(y_disp_test, preds))

In [ ]:
disp_preds = disp_xgb.predict(test_df_disp)
test['disposition'] = disp_preds

In [ ]:
test

References:

In [ ]:
#For ed_los_values (To Estimate Hidden Continuous Value Regimes)
# --- ed_los_hours ---
train_df_ed = train.copy()

X_ed = train_df_ed.drop(columns=['triage_acuity', 'disposition', 'ed_los_hours'], axis=1)
y_ed = train_df_ed['ed_los_hours']

X_ed_train, X_ed_test, y_ed_train, y_ed_test = train_test_split(X_ed, y_ed, test_size=0.2, random_state=42)

ed_lgb = LGBMRegressor(
    objective='regression',
    learning_rate=0.03,
    n_estimators=2000,
    subsample=0.8,
    colsample_bytree=0.8,
    num_leaves=50,
    min_child_samples=30,
    reg_alpha=0.6,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1
)

ed_lgb.fit(
    X_ed_train, y_ed_train
)

preds = disp_xgb.predict(X_ed_test)
print(f"Model Metric Results: \n")
print(f"MSE: {mean_squared_error(y_ed_test, preds)}")
print(f"MAE: {mean_absolute_error(y_ed_test, preds)}")

In [ ]:
test_df_ed = test_df_disp.copy()
ed_preds = ed_lgb.predict(test_df_ed)
test["ed_los_hours"] = ed_preds

In [ ]:
test

## Now PCA & LDA for Hidden Regimes in Data

In [ ]:
train_df = train.copy()
test_df = test.copy()

In [ ]:
train_df

In [ ]:
test_df

In [ ]:
ensemble_pipe = Pipeline([
    ('scaler', RobustScaler()),
    ('pca', PCA(n_components=0.95)),
    ('lda', LinearDiscriminantAnalysis(solver='eigen'))
])

X = train_df.drop(columns=['triage_acuity'], axis=1)
y = train_df['triage_acuity']

temperature = 0.05

In [ ]:
X.shape

In [ ]:
ensemble_pipe.fit(X, y)

# Ensure test_df has the same columns in the same order as X
test_df_aligned = test[X.columns]

ensemble_preds = ensemble_pipe.predict(test_df_aligned)
ensemble_probs = ensemble_pipe.predict_proba(test_df_aligned)

logits = np.log(ensemble_probs + 1e-9)
scaled_logits = logits / temperature
probs = softmax(scaled_logits)

test['hidden_regime'] = ensemble_preds
# Assign each probability column to a new DataFrame column
for i in range(probs.shape[1]):
    test[f'hidden_regime_prob_{i+1}'] = probs[:, i]

In [ ]:
train_preds = ensemble_pipe.predict(X)
train_probs = ensemble_pipe.predict_proba(X) * 0.05 #alpha for penalty in probability scoring

train_logits = np.log(train_probs + 1e-9)
train_scaled_logits = train_logits / temperature
train_probs = softmax(train_scaled_logits)


train['hidden_regime'] = train_preds
for i in range(train_probs.shape[1]):
    train[f'hidden_regime_prob_{i+1}'] = train_probs[:, i]

In [ ]:
train

In [ ]:
test

In [ ]:
train.columns

In [ ]:
test.columns

In [ ]:
#Regime Confidence (To Prevent Model Leakage)
train['hr_z_regime'] = (
    train['heart_rate'] - train.groupby('hidden_regime')['heart_rate'].transform('mean')
) / train.groupby('hidden_regime')['heart_rate'].transform('std')

test['hr_z_regime'] = (
    test['heart_rate'] - test.groupby('hidden_regime')['heart_rate'].transform('mean')
) / test.groupby('hidden_regime')['heart_rate'].transform('std')


train['regime_confidence'] = train[[f'hidden_regime_prob_{i}' for i in range(1,6)]].max(axis=1)
train['regime_entropy'] = -np.sum(
    train[[f'hidden_regime_prob_{i}' for i in range(1,6)]].values *
    np.log(train[[f'hidden_regime_prob_{i}' for i in range(1,6)]].values + 1e-9),
    axis=1
)

test['regime_confidence'] = test[[f'hidden_regime_prob_{i}' for i in range(1,6)]].max(axis=1)
test['regime_entropy'] = -np.sum(
    test[[f'hidden_regime_prob_{i}' for i in range(1,6)]].values *
    np.log(test[[f'hidden_regime_prob_{i}' for i in range(1,6)]].values + 1e-9),
    axis=1
)

#Relative Features
train_means = train[['heart_rate','spo2','respiratory_rate']].mean()

train['hr_dev'] = train['heart_rate'] - train_means['heart_rate']
train['spo2_dev'] = train['spo2'] - train_means['spo2']

test_means = test[['heart_rate','spo2','respiratory_rate']].mean()

test['hr_dev'] = test['heart_rate'] - test_means['heart_rate']
test['spo2_dev'] = test['spo2'] - test_means['spo2']

#Cross Domain Feature Interactions
train['age_hr'] = train['age'] * train['heart_rate']
train['comorbidity_spo2'] = train['num_comorbidities'] * train['spo2']

test['age_hr'] = test['age'] * test['heart_rate']
test['comorbidity_spo2'] = test['num_comorbidities'] * test['spo2']

#Decision Boundary Features (To help The assumptions of Doctors)
train['critical_flag'] = (
    (train['spo2'] < 92) |
    (train['heart_rate'] > 120) |
    (train['gcs_total'] < 15)
).astype(int)

test['critical_flag'] = (
    (test['spo2'] < 92) |
    (test['heart_rate'] > 120) |
    (test['gcs_total'] < 15)
).astype(int)


In [ ]:
train

In [ ]:
test

In [ ]:
#Removing Disposition and ed_los_hours from training and testing as it's pupose was to only extract hidden regimes
train = train.drop(columns=['disposition', 'ed_los_hours'], axis=1)
test = test.drop(columns=['disposition', 'ed_los_hours'], axis=1)

In [ ]:
train

In [ ]:
test

## Modelling

In [ ]:
class WeightedGeometricEnsemble(BaseEstimator, ClassifierMixin):
    def __init__(self, estimators=None, weights=None, alpha=0.05, lambda_corr=0.1):
        self.estimators = estimators
        self.weights = weights
        self.alpha = alpha
        self.lambda_corr = lambda_corr

    def fit(self, X, y):
        self.fitted_estimators_ = []
        self.calibrators_ = []
        self.classes_ = np.unique(y)

        weights_arr = np.array(self.weights)
        base_preds = []

        # 1. Fit base models + isotonic calibration
        for est in self.estimators:
            model = clone(est)
            model.fit(X, y)
            self.fitted_estimators_.append(model)

            # For multiclass, we simplify to the probability of the positive class if binary
            # or handle multiple columns. Here we assume we need probabilities for all samples
            probs = model.predict_proba(X)

            # Simplified calibration for demonstration: using the argmax probability
            confidences = np.max(probs, axis=1)

            iso = IsotonicRegression(out_of_bounds="clip")
            # Isotonic expects 1D, we use the max prob vs correctness proxy or simplify to binary logic
            # For true multiclass, this needs a loop per class, but keeping it simple for the clone fix:
            iso.fit(confidences, (model.predict(X) == y).astype(int))

            self.calibrators_.append(iso)
            base_preds.append(confidences)

        base_preds = np.array(base_preds)
        corr_matrix = np.corrcoef(base_preds)
        corr_penalty = np.sum(np.abs(corr_matrix)) - len(corr_matrix)

        weights_norm = weights_arr / np.sum(weights_arr)
        log_preds = np.log(np.clip(base_preds, 1e-6, 1 - 1e-6))
        weighted_log = np.sum(weights_norm[:, None] * log_preds, axis=0)
        geo_preds = np.exp(weighted_log)

        stacked_features = np.array([est.predict_proba(X).ravel() for est in self.fitted_estimators_]).T
        # Reshape to (n_samples, n_models * n_classes)
        n_samples = X.shape[0]
        n_models = len(self.fitted_estimators_)
        n_classes = len(self.classes_)
        stacked_features = np.zeros((n_samples, n_models * n_classes))

        for i, est in enumerate(self.fitted_estimators_):
            stacked_features[:, i*n_classes : (i+1)*n_classes] = est.predict_proba(X)

        self.meta_model_ = LogisticRegression(max_iter=1000)
        self.meta_model_.fit(stacked_features, y)

        return self

    def predict_proba(self, X):
        n_samples = X.shape[0]
        n_models = len(self.fitted_estimators_)
        n_classes = len(self.classes_)

        stacked_features = np.zeros((n_samples, n_models * n_classes))
        base_probs_list = []

        for i, est in enumerate(self.fitted_estimators_):
            p = est.predict_proba(X)
            stacked_features[:, i*n_classes : (i+1)*n_classes] = p
            base_probs_list.append(p)

        # Meta-model predictions
        meta_probs = self.meta_model_.predict_proba(stacked_features)

        # Geometric mean of probabilities per class
        weights_norm = np.array(self.weights) / np.sum(self.weights)
        # Shape: (n_models, n_samples, n_classes)
        base_probs_arr = np.array(base_probs_list)

        # Log-space weighted sum
        log_probs = np.log(np.clip(base_probs_arr, 1e-6, 1 - 1e-6))
        weighted_log = np.sum(weights_norm[:, None, None] * log_probs, axis=0)
        geo_probs = np.exp(weighted_log)
        # Re-normalize geo_probs
        geo_probs = geo_probs / geo_probs.sum(axis=1, keepdims=True)

        # Final blend
        final_probs = (1 - self.alpha) * geo_probs + self.alpha * meta_probs
        return final_probs

    def predict(self, X):
        return np.argmax(self.predict_proba(X), axis=1)

## The Weighted Geometric Ensemble Formula

$$pfinal​(x)=(1−α)(i=1∏M​p^​i​(x)w~i​)+α⋅σ(i=1∑M​θi​p^​i​(x))$$

## With The Training Objective:
$$ min[LogLoss(y,pfinal​(x))+λi=j∑​∣corr(p^​i​,p^​j​)∣]​$$

#Where:
$$logpgeo​(x)=i=1∑M​w~i​logpi​(x)$$
- Prediction consensus under multiplicative belief agreement

## With Isotonic Calibration Per Model

Each model output becomes:
$$p^​i​(x)=Isotonic(pi​(x))$$

and penalizes Model Correlation by:
$$R=\sum_{i!=j}{​∣Cij​∣}$$
Where $Cij$ is $$Cij​=corr(p^​i​,p^​j​)$$
- Where we reward accuracy, penalize redundant opinions

## With The addition of Logit Blending Layer wherin:
$$pfinal​(x)=(1−α)⋅pgeo​(x)+α⋅σ(g(p^​1​(x),...,p^​M​(x)))$$

where:
- $𝑔$ = logistic regression meta-model
- $σ$ = probability output

In [ ]:
#Base Models Layer
xgb = XGBClassifier(
    n_estimators=1000,
    num_class=5,
    learning_rate=0.01,
    max_depth=6,
    min_child_weight=5,
    gamma=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    objective='multi:softprob',
    eval_metric='mlogloss',
    tree_method='hist',
    random_state=42,
    device="gpu",
    n_jobs=-1
)
lgb =  LGBMClassifier(
    n_estimators=1500,
    learning_rate=0.01,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=20,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    objective='multiclass',  # or 'binary'
    metric='multi_logloss',
    random_state=42,
    device="gpu",
    n_jobs=-1
)
rf = RandomForestClassifier(
    n_estimators=500,
    max_depth=12,
    min_samples_split=10,
    min_samples_leaf=5,
    max_features='sqrt',
    bootstrap=True,
    random_state=42,
    n_jobs=-1
)

nb = Pipeline([
    ('scaler', MinMaxScaler()),
    ('model', GaussianNB(var_smoothing=1e-9))
])

lr = Pipeline([
    ('scaler', RobustScaler()),
    ('model',  LogisticRegression(
      penalty='l2',
      C=1.0,
      max_iter=1000,
      solver='lbfgs',
      multi_class='multinomial',
      random_state=42,
      n_jobs=-1
  ))
])

#Normal Ensembled Layer
voting = Pipeline([
    ('model', VotingClassifier(
      estimators=[("xgb", xgb), ("lgb", lgb), ("rf", rf), ("lr", lr)],
      voting='soft',
      weights=[3, 5, 2, 1],
      n_jobs=-1
    ))
])

stacking = Pipeline([
    ('model', StackingClassifier(
      estimators=[
          ('xgb', xgb),
          ('lgb', lgb),
          ('rf', rf),
          ('lr', lr)
      ],
      final_estimator=LogisticRegression(
          penalty='l2',
          C=1.0,
          max_iter=1000
      ),
      passthrough=True,
      cv=5,
      n_jobs=-1
  ))
])

#Final (Proposed) Ensembled Layer
wgb = WeightedGeometricEnsemble(
    estimators=[xgb, lgb, rf, lr],
    weights=[3, 5, 2, 1],
    alpha=0.05,
    lambda_corr=0.1
)

In [ ]:
X = train.drop(columns=['triage_acuity'], axis=1)
y = train['triage_acuity']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

smote = SMOTE(random_state=42)
X_train, y_train = smote.fit_resample(X_train, y_train)

In [ ]:
models = {
    'XGBClassifier': xgb,
    'LGBCLassifier': lgb,
    'RandomForestClassifier': rf,
    'GaussianNB': nb,
    'LogisticRegression': lr,
    'VotingClassifier': voting,
    'StackingClassifier': stacking,
    'WeightedGeometricEnsemble': wgb
}

mlflow.set_experiment("OOF Predictions with Models")

y_shifted = y - 1

n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

SHOW_PLOTS = True
num_classes = len(np.unique(y_shifted))

for model_name, model in models.items():
    print("\n" + "="*60)
    print(f"MODEL: {model_name}")
    print("="*60)

    oof_preds = np.zeros((len(X), num_classes))
    fold_losses = []

    with mlflow.start_run(run_name=model_name):
        for fold, (train_idx, val_idx) in enumerate(skf.split(X, y_shifted)):
            X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_train, y_val = y_shifted.iloc[train_idx], y_shifted.iloc[val_idx]

            m = clone(model)
            m.fit(X_train, y_train)

            preds = m.predict_proba(X_val)
            oof_preds[val_idx] = preds

            loss = log_loss(y_val, preds)
            fold_losses.append(loss)
            print(f"Fold {fold} LogLoss: {loss:.5f}")
            mlflow.log_metric(f"fold_{fold}_logloss", loss)

        oof_class = np.argmax(oof_preds, axis=1)
        final_logloss = log_loss(y_shifted, oof_preds)
        final_acc = accuracy_score(y_shifted, oof_class)
        final_auc = roc_auc_score(y_shifted, oof_preds, multi_class='ovr', average='weighted')

        print("\n--- FINAL OOF METRICS ---")
        print(f"LogLoss:   {final_logloss:.5f}")
        print(f"Accuracy:  {final_acc:.5f}")
        print(f"AUC:       {final_auc:.5f}")

        mlflow.log_metric("oof_logloss", final_logloss)
        mlflow.log_metric("oof_accuracy", final_acc)
        mlflow.log_metric("oof_auc", final_auc)

        cm = confusion_matrix(y_shifted, oof_class)
        disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=np.unique(y))
        disp.plot()
        plt.title(f"{model_name} - Confusion Matrix")
        plt.show()

        fig, ax = plt.subplots(figsize=(8, 6))
        for i in range(num_classes):
            y_true_binary = (y_shifted == i).astype(int)
            RocCurveDisplay.from_predictions(y_true_binary, oof_preds[:, i], name=f"Class {i+1} vs Rest", ax=ax)
        plt.title(f"{model_name} - ROC Curves")
        plt.show()

        mlflow.set_tag("model_name", model_name)

## Global Variable for all tests

In [ ]:
def get_preds_dict(models, X):
    return {name: model.predict_proba(X)for name, model in models.items()}
models = {
    "xgb": xgb,
    "lgb": lgb,
    "rf": rf,
    "nb": nb,
    "lr": lr,
    "voting": voting,
    "stacking": stacking,
    "wgb": wgb
}


for model in models.values():
    model.fit(X_train, y_train)


preds_dict = get_preds_dict(models, X_test)



## Statistical Testing

In [ ]:
def wilcoxon_signed_rank(a, b):
    return stats.wilcoxon(a, b)
def wilcoxon_rank_sum(a, b):
    return stats.mannwhitneyu(a, b, alternative='two-sided')
def cliffs_delta(a, b):
    a = np.asarray(a).flatten()
    b = np.asarray(b).flatten()

    greater = 0
    less = 0

    for i in a:
        for j in b:
            if i > j:
                greater += 1
            elif i < j:
                less += 1

    return (greater - less) / (len(a) * len(b))
def nemenyi_test(ranks):
    ranks = np.array(ranks)
    k = ranks.shape[1]
    n = ranks.shape[0]

    avg_ranks = np.mean(ranks, axis=0)

    cd = 2.0 * np.sqrt(k * (k + 1) / (6.0 * n))

    comparisons = []
    for i in range(k):
        for j in range(i + 1, k):
            diff = abs(avg_ranks[i] - avg_ranks[j])
            comparisons.append((i, j, diff > cd))

    return avg_ranks, cd, comparisons
def mcnemar_test(y_true, pred_a, pred_b):
    pred_a = np.argmax(pred_a, axis=1)
    pred_b = np.argmax(pred_b, axis=1)

    a_correct = pred_a == y_true
    b_correct = pred_b == y_true

    a_both_correct = np.sum(a_correct & b_correct)
    a_only = np.sum(a_correct & ~b_correct)
    b_only = np.sum(~a_correct & b_correct)

    table = np.array([[a_both_correct, a_only],
                      [b_only, np.sum(~a_correct & ~b_correct)]])

    return table

    
def bootstrap_test(y_true, pred_a, pred_b, metric_fn, n_bootstrap=1000):

    y_true = np.array(y_true)
    pred_a = np.array(pred_a)
    pred_b = np.array(pred_b)
    
    diffs = []

    for _ in range(n_bootstrap):
        idx = resample(np.arange(len(y_true)))
        y_sample = y_true[idx]
        a_sample = pred_a[idx]
        b_sample = pred_b[idx]

        diff = log_loss(y_sample, a_sample, labels=[0,1,2,3,4]) - log_loss(y_sample, b_sample, labels=[0,1,2,3,4])
        diffs.append(diff)

    lower = np.percentile(diffs, 2.5)
    upper = np.percentile(diffs, 97.5)

    return lower, upper


In [ ]:
#Usage
def per_sample_log_loss(y_true, p_pred, labels):
    y_true = np.array(y_true)
    p_pred = np.array(p_pred)

    losses = []
    for i in range(len(y_true)):
        true_class = y_true[i]
        if true_class not in labels:
            raise ValueError(f"Label {true_class} not in provided labels {labels}")

        class_index = labels.index(true_class)
        prob = p_pred[i, class_index]
        losses.append(-np.log(prob + 1e-15))

    return np.array(losses)
    
def statistical_testing_pipeline(y_true, preds_dict):
    """
    preds_dict: {
        "model_a": preds_a,
        "model_b": preds_b,
        ...
    }
    """
    y_true = y_true - 1
    results = {}

    models = list(preds_dict.keys())

    for i in range(len(models)):
        for j in range(i + 1, len(models)):

            m1, m2 = models[i], models[j]
            p1, p2 = preds_dict[m1], preds_dict[m2]

            loss1 = per_sample_log_loss(y_true, p1, labels=[0,1,2,3,4])
            loss2 = per_sample_log_loss(y_true, p2, labels=[0,1,2,3,4])

            results[f"{m1}_vs_{m2}"] = {
                "wilcoxon_signed_rank": wilcoxon_signed_rank(loss1, loss2),
                "wilcoxon_rank_sum": wilcoxon_rank_sum(loss1, loss2),
                "cliffs_delta": cliffs_delta(loss1, loss2),
                "bootstrap_ci": bootstrap_test(y_true, p1, p2, metric_fn=log_loss),
                "mcnemar": mcnemar_test(
                    y_true,
                    (p1 > 0.5).astype(int),
                    (p2 > 0.5).astype(int)
                )
            }

    return results

In [ ]:
stat_results = statistical_testing_pipeline(y_test, preds_dict)
print(pd.DataFrame(stat_results))

In [ ]:
pd.DataFrame(stat_results).to_csv("Statistics_result.csv", index=False)

## Stress Testing

In [ ]:
def feature_occlusion(model, X, y, metric_fn=log_loss):
    X = np.array(X)
    y = np.array(y)

    probs = model.predict_proba(X)
    baseline = metric_fn(y, probs, labels=np.unique(y))

    results = {}

    for i in range(X.shape[1]):
        X_perturbed = X.copy()
        X_perturbed[:, i] = np.random.permutation(X_perturbed[:, i])

        probs_perturbed = model.predict_proba(X_perturbed)
        score = metric_fn(y, probs_perturbed, labels=np.unique(y))

        results[f"feature_{i}"] = score - baseline

    return results

In [ ]:
def noise_occlusion(model, X, y, metric_fn=log_loss, noise_std=0.1):
    X = np.array(X)
    y = np.array(y)

    probs = model.predict_proba(X)
    baseline = metric_fn(y, probs, labels=np.unique(y))

    noise = np.random.normal(0, noise_std, X.shape)
    X_noisy = X + noise

    noisy_probs = model.predict_proba(X_noisy)
    noisy_score = metric_fn(y, noisy_probs, labels=np.unique(y))

    return baseline, noisy_score

In [ ]:
def stress_testing_pipeline(model, X, y):

    results = {}

    # Feature occlusion
    results["feature_occlusion"] = feature_occlusion(
        model, X, y, metric_fn=log_loss
    )

    # Noise occlusion
    baseline, noisy = noise_occlusion(
        model, X, y, metric_fn=log_loss
    )

    results["noise_occlusion"] = {
        "baseline": baseline,
        "noisy": noisy
    }

    return results

In [ ]:
for model_name, model in models.items():
    print(f"Model: {model_name}")
    stress_results = stress_testing_pipeline(model, X, y)
    print(stress_results)

##Impossibility Testing

In [ ]:
def label_shuffle_test(model, X, y):
    y = np.array(y)
    X = np.array(X)

    y_shuffled = np.random.permutation(y)


    model_clone = clone(model)
    model_clone.fit(X, y_shuffled)

    probs = model_clone.predict_proba(X)

    return log_loss(y_shuffled, probs, labels=np.unique(y))

In [ ]:
def feature_randomization_test(model, X, y, col):
    X = np.array(X)
    y = np.array(y)

    X_rand = X.copy()
    np.random.shuffle(X_rand[:, col])

    probs = model.predict_proba(X_rand)

    return log_loss(y, probs, labels=np.unique(y))

In [ ]:
def impossibility_testing_pipeline(model, X, y):

    results = {}
    y = y - 1
    # Label shuffle
    results["label_shuffle_loss"] = label_shuffle_test(model, X, y)

    # Feature randomization
    feature_losses = {}
    for i in range(X.shape[1]):
        loss = feature_randomization_test(model, X, y, col=i)
        feature_losses[f"feature_{i}"] = loss

    results["feature_randomization"] = feature_losses

    return results

In [ ]:
for model_name, model in models.items():
    print(f"Model: {model_name}")
    impossibility_results = impossibility_testing_pipeline(model, X, y)
    print(impossibility_results)

## Causal Inference

In [ ]:
def econml_dml(X, T, Y):
    from econml.dml import LinearDML
    from sklearn.linear_model import LassoCV

    est = LinearDML(model_y=LassoCV(), model_t=LassoCV())
    est.fit(Y, T, X=X)

    return est.effect(X)

In [ ]:
def causal_inference_dowhy(data, treatment, outcome, graph):

    data = data.copy()
    data.columns = data.columns.str.strip()

    # Ensure no duplicates
    data = data.loc[:, ~data.columns.duplicated()]

    from dowhy import CausalModel

    model = CausalModel(
        data=data,
        treatment=treatment,
        outcome=outcome,
        graph=graph
    )

    identified_estimand = model.identify_effect()

    estimate = model.estimate_effect(
        identified_estimand,
        method_name="backdoor.linear_regression"
    )

    if estimate is None or estimate.value is None:
        return estimate, None

    try:
        refute = model.refute_estimate(
            identified_estimand,
            estimate,
            method_name="placebo_treatment_refuter"
        )
    except Exception as e:
        print("Refutation failed:", e)
        refute = None

    return estimate, refute

In [ ]:
treatment = "heart_rate"
outcome = "triage_acuity"

graph = """
digraph {

    age -> heart_rate;
    age -> triage_acuity;
    age -> critical_flag;

    num_comorbidities -> heart_rate;
    num_comorbidities -> triage_acuity;
    num_comorbidities -> critical_flag;

    systolic_bp -> heart_rate;
    respiratory_rate -> heart_rate;
    temperature_c -> heart_rate;
    spo2 -> heart_rate;

    heart_rate -> triage_acuity;
    triage_acuity -> critical_flag;

}
"""

X = train.drop(columns=[outcome])
T = train[treatment]
Y = train[outcome]


T = train["heart_rate"].copy()
T.name = "heart_rate"

causal_results = causal_inference_dowhy(
    data=train,
    treatment="heart_rate",
    outcome="critical_flag",
    graph=graph
)
causal_results

In [ ]:
estimate, refute = causal_results

print("Causal Effect (ATE):", estimate.value)

In [ ]:
print(estimate.target_estimand)

In [ ]:
print(refute)

In [ ]:
print(estimate)

## Causal Conformal Prediction

In [ ]:
def get_expected_value(preds):
    classes = np.arange(preds.shape[1])
    return preds @ classes

In [ ]:
def conformal_prediction(residuals, alpha=0.1):
    residuals = np.abs(residuals)
    q = np.quantile(residuals, 1 - alpha)
    return q

def conformal_interval(pred, residual_quantile):
    lower = pred - residual_quantile
    upper = pred + residual_quantile
    return lower, upper

In [ ]:
for name, preds in preds_dict.items():
    preds = np.array(preds)
    y_true = np.array(y_test) - 1

    probs = preds[np.arange(len(y_true)), y_true]
    residuals = 1 - probs

    q = conformal_prediction(residuals)

    prediction_sets = preds >= (1 - q)

    print(f"Model: {name}")
    print(f"Quantile (q): {q}")
    print(f"Example prediction set[0]: {prediction_sets[0]}")

## Uplift Modelling

In [ ]:
T = train["heart_rate"]  # treatment
y = train["triage_acuity"] - 1  # outcome
X = train.drop(columns=["triage_acuity", "heart_rate"])

X_train, X_test, y_train, y_test, T_train, T_test = train_test_split(
    X, y, T, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
treated_models = {}
control_models = {}

T_train_binary = (T_train > T_train.median()).astype(int)
T_test_binary  = (T_test > T_train.median()).astype(int)

treated_idx = (T_train_binary == 1)
control_idx = (T_train_binary == 0)

X_treated = X_train.loc[treated_idx]

y_treated = y_train[treated_idx]

X_control = X_train[control_idx]
y_control = y_train[control_idx]


for name, model in models.items():
    model_t = clone(model)
    model_c = clone(model)

    model_t.fit(X_treated, y_treated)
    model_c.fit(X_control, y_control)

    treated_models[name] = model_t
    control_models[name] = model_c

In [ ]:
def two_model_uplift(model_t, model_c, X):
    p_t = model_t.predict_proba(X)
    p_c = model_c.predict_proba(X)

    # Binary case
    if p_t.shape[1] == 2:
        return p_t[:, 1] - p_c[:, 1]

    # Multiclass case (expected value)
    classes = model_t.classes_
    return (p_t * classes).sum(axis=1) - (p_c * classes).sum(axis=1)

In [ ]:
uplift_scores = {}

for name in models.keys():
    uplift_scores[name] = two_model_uplift(
        treated_models[name],
        control_models[name],
        X_test
    )

In [ ]:
print(pd.DataFrame(uplift_scores))

##Model Interpretability

In [ ]:
def permutation_importance_analysis(model, X, y, scoring='neg_log_loss', n_repeats=10):
    result = permutation_importance(
        model,
        X,
        y,
        scoring=scoring,
        n_repeats=n_repeats,
        random_state=42
    )

    return result.importances_mean, result.importances_std

In [ ]:
import shap
def shap_analysis(model, X):
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X)

    return shap_values

In [ ]:
def shap_summary_plot(name, shap_values, X):
    shap.summary_plot(shap_values, X)
    plt.savefig(f"Summary_plot_{name}")
    plt.show()

In [ ]:
def lime_explanation(model, X_train, X_test_instance):
    explainer = LimeTabularExplainer(
        training_data=X_train,
        mode='classification'
    )

    exp = explainer.explain_instance(
        X_test_instance,
        model.predict_proba
    )

    return exp

In [ ]:
X = train.drop(columns=['triage_acuity'], axis=1)
y = train['triage_acuity']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

smote = SMOTE(random_state=42)
X_train, y_train = smote.fit_resample(X_train, y_train)

for model_name, model in models.items():
    print(f"Model: {model_name}")
    
    X_eval = X_test.copy()
    X_eval = X_eval[model.feature_names_in_]
    
    print("Permutation Importance:")
    print(permutation_importance_analysis(model, X_eval, y_test))
    
    print("SHAP Analysis:")
    print(shap_analysis(model, X_eval))

## A/B Testing

In [ ]:
def ab_test(y_true, preds_a, preds_b):
    y_true = np.array(y_true) - 1
    # log loss for multiclass (pick probability of the true class)
    loss_a = -np.log(preds_a[np.arange(len(y_true)), y_true])
    loss_b = -np.log(preds_b[np.arange(len(y_true)), y_true])

    stat, p_value = stats.ttest_rel(loss_a, loss_b)

    return stat, p_value

In [ ]:
def ab_bootstrap(y_true, preds_a, preds_b, metric_fn, n_bootstrap=1000):
    # Convert to numpy
    y_true = np.array(y_true)
    preds_a = np.array(preds_a)
    preds_b = np.array(preds_b)

    scores_diff = []

    for _ in range(n_bootstrap):
        idx = np.random.choice(len(y_true), len(y_true), replace=True)

        score_a = metric_fn(y_true[idx], preds_a[idx])
        score_b = metric_fn(y_true[idx], preds_b[idx])

        scores_diff.append(score_a - score_b)

    lower = np.percentile(scores_diff, 2.5)
    upper = np.percentile(scores_diff, 97.5)

    return lower, upper

In [ ]:
#Mcnemar Style A/B Testing
def ab_mcnemar(y_true, pred_a, pred_b):
    correct_a = pred_a == y_true
    correct_b = pred_b == y_true

    b = np.sum(correct_a & ~correct_b)
    c = np.sum(~correct_a & correct_b)

    stat = (abs(b - c) - 1)**2 / (b + c + 1e-9)
    p_value = 1 - stats.chi2.cdf(stat, df=1)

    return stat, p_value

In [ ]:
from itertools import combinations

model_names = list(models.keys())

for model_a, model_b in combinations(model_names, 2):
    print(f"\nA/B Test: {model_a} vs {model_b}")
    
    preds_a = preds_dict[model_a]
    preds_b = preds_dict[model_b]
    
    # t-test
    stat, p_value = ab_test(y_test, preds_a, preds_b)
    print(f"t-test: stat={stat}, p-value={p_value}")
    
    # Bootstrap CI
    lower, upper = ab_bootstrap(
        y_test,
        preds_a,
        preds_b,
        metric_fn=log_loss
    )
    print(f"Bootstrap CI: [{lower}, {upper}]")